In [13]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [30]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.rwf2000 import RWF2000Dataset
from src.baseline_cnn_lstm import BaselineCNNLSTM
from src.config import DATASET_ROOT
from src.config import CHECKPOINT_DIR
from tqdm.notebook import tqdm

ImportError: cannot import name 'CHECKPOINT_DIR' from 'src.config' (/mnt/vurm/homes/homes/mp2940/violence-detection-dissertation/src/config.py)

In [22]:
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print(device)

cuda:2


In [23]:
# dataset and loaders

train_dataset = RWF2000Dataset(DATASET_ROOT, split="train", num_frames=16)
val_dataset = RWF2000Dataset(DATASET_ROOT, split="val", num_frames=16)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=4
)

In [24]:
model = BaselineCNNLSTM(
    hidden_size=256,
    num_layers=1,
    num_classes=2,
    dropout=0.3,
    freeze_cnn=True
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [25]:
# training function

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    
    for videos, labels in dataloader:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        outputs = model(videos)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * videos.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{correct/total:.4f}"
        )
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [26]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for videos, labels in dataloader:
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(videos)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * videos.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{correct/total:.4f}"
            )
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [27]:
num_epochs = 10

for epoch in range(num_epochs):

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )


Epoch 1/10
--------------------------------------------------


Training:   0%|          | 0/800 [00:00<?, ?it/s]

Validation:   0%|          | 0/200 [00:00<?, ?it/s]

Train Loss: 0.6553 | Train Acc: 0.6169 | Val Loss: 0.5942 | Val Acc: 0.7100

Epoch 2/10
--------------------------------------------------


Training:   0%|          | 0/800 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/200 [00:00<?, ?it/s]

Train Loss: 0.5631 | Train Acc: 0.6950 | Val Loss: 0.5917 | Val Acc: 0.7100

Epoch 10/10
--------------------------------------------------


Training:   0%|          | 0/800 [00:00<?, ?it/s]

Validation:   0%|          | 0/200 [00:00<?, ?it/s]

Train Loss: 0.5629 | Train Acc: 0.6863 | Val Loss: 0.5765 | Val Acc: 0.6950


In [28]:
model.state_dict()

OrderedDict([('cnn.0.0.weight',
              tensor([[[[-6.3108e-02, -1.8766e-01, -1.5188e-01],
                        [-4.9379e-01, -6.4248e-01, -5.8935e-01],
                        [-6.8005e-01, -9.7448e-01, -7.6317e-01]],
              
                       [[-1.6350e-02, -1.8482e-02,  6.2783e-02],
                        [ 3.5436e-02,  5.8980e-02,  1.0693e-01],
                        [ 1.6995e-01,  1.4699e-01,  1.8521e-01]],
              
                       [[ 1.1395e-01,  1.6316e-01,  1.0483e-01],
                        [ 4.0824e-01,  5.7489e-01,  4.7270e-01],
                        [ 5.7547e-01,  7.1503e-01,  5.3702e-01]]],
              
              
                      [[[ 2.9983e-03,  1.4297e-02,  5.9918e-02],
                        [ 5.5779e-03,  3.0330e-02, -4.5050e-02],
                        [ 1.2600e-01,  5.5075e-02, -9.2239e-01]],
              
                       [[ 4.2316e-03, -6.6995e-02, -7.4151e-02],
                        [ 1.4910e-02,  1.56

In [33]:
from pathlib import Path

CHECKPOINT_DIR = Path("/homes/mp2940/violence-detection-dissertation/checkpoints")

In [34]:
torch.save(
    model.state_dict(),
    CHECKPOINT_DIR / "baseline_cnn_lstm" / "baseline_cnn_lstm_v1.pth" 
)